# The J/ψ peak from CMS open data

The J/ψ is a bound state of a charm quark and its antiquark, far lighter than the Z. Its natural width is so small that the
width of its peak comes almost entirely from the detector, which makes it a direct view of the muon momentum resolution.

In [ ]:
import json
import os
import re
from pathlib import Path

from IPython.display import Image, Markdown, display

import cmsopen
from cmsopen.fitting import pdg_mass_width
from cmsopen.pipeline import CHANNELS

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "examples" / "data" / "cmsopen_sample.root").exists())
SAMPLE = ROOT / "examples" / "data" / "cmsopen_sample.root"
FIGURES = ROOT / "docs" / "figures"
PRODUCTION = json.loads((ROOT / "results" / "production.json").read_text())
print(f"cmsopen {cmsopen.__version__} imported from: {cmsopen.__file__}")


class Checks:
    """Collects self-check results and prints one OK/FAIL line each."""

    def __init__(self):
        self.failures = []

    def check(self, name, ok, detail=""):
        """Print one OK or FAIL line and remember failures."""
        print(f"{'OK' if ok else 'FAIL'}: {name}" + (f" ({detail})" if detail else ""))
        if not ok:
            self.failures.append(name)


checks = Checks()

## Analyse the sample

The J/ψ channel has its own trigger and softer muon cuts, because its muons have much lower momentum than those from a
Z. The sample is small, so we fit `voigt_poly` here; the numbers below are a demonstration and are not results.

In [ ]:
config = CHANNELS["Jpsi_mumu"]
print(config.description)
result = cmsopen.analyze(SAMPLE, channel="Jpsi_mumu", model="voigt_poly", stability=False, verbose=False)
print(result.fit.table())
display(result.figures["mass_fit"])

In [ ]:
pdg_mass = pdg_mass_width(config.resonance)[0]
peak, error = result.fit.peak_position
n_pairs = result.fit.n_entries
checks.check("the sample gives a fit with pairs in its range", n_pairs > 500, f"{n_pairs:,} pairs")
checks.check("the peak is within a generous window of the PDG mass", abs(peak - pdg_mass) < 0.02, f"{peak:.4f} GeV")

## What the frozen model does on so few pairs

The production model has two resolution cores, and the sample may not have enough pairs to determine the second one.
The library either fits it or fails with a clear `AnalysisError`, and it does not return a fit it cannot trust.

In [ ]:
try:
    frozen = cmsopen.analyze(SAMPLE, channel="Jpsi_mumu", stability=False, verbose=False)
    outcome, ok = f"fitted: {frozen.fit.model_name}", frozen.fit.model_name == config.model
except cmsopen.AnalysisError as exc:
    outcome, ok = f"AnalysisError: {str(exc)[:110]}", "fit failed" in str(exc)
print(outcome)
checks.check("the frozen default fits or fails with the documented AnalysisError", ok)

## Optional: a whole file

If `CMSOPEN_DATA_FILE` points to a NanoAOD file, this cell runs the frozen default model on it.

In [ ]:
data_file = os.environ.get("CMSOPEN_DATA_FILE")
if data_file:
    full = cmsopen.analyze(data_file, channel="Jpsi_mumu", stability=False, verbose=False)
    print(full.fit.table())
    display(full.figures["mass_fit"])
else:
    print("CMSOPEN_DATA_FILE is not set: skipped")

## Production results

These come from the single run on the pre-registered sample. A benchmark is a requirement fixed before the sample was opened; the
observed values are what was measured.



Measured on 20 files (controlling **benchmark**: 20 pre-registered files, met).

| J/ψ → μ⁺μ⁻ | value |
|---|---|
| Model and fit range | `cb2_bw_poly2`, 2.8-3.4 GeV |
| Events read, pairs in the fit range | 9,780,165, 402,920 |
| Peak (pole) position, observed | 3.09332 GeV |
| Statistical error, raw | 0.00007 GeV |
| Statistical error, scaled by S = 1.740 | 0.00013 GeV |
| Resolution, observed | 0.02712 ± 0.00012 GeV |
| χ²/ndf **benchmark** | ≤ 2.0 |
| χ²/ndf, observed | 63.6 / 21 = 3.028 (not met) |
| PDG mass (reference only) | 3.0969 GeV |
| Offset from PDG, observed | -0.00358 GeV |
| Fit-choice spread, observed | 0.00006 GeV |

This peak lies below the PDG mass. No momentum-scale corrections are applied, so this is not a measurement of the
J/ψ mass. Compare the statistical error, raw and scaled: when χ²/ndf is above one, the scaled error is the more honest of the two.

In [ ]:
NOTEBOOK = "02_jpsi_peak.ipynb"
NOTEBOOK_CHANNELS = ("Jpsi_mumu",)
NOTEBOOK_FIGURES = ("jpsi_peak_fit.png", "jpsi_os_vs_ss.png")
# Structural self-checks: every number stated in the "Production results" text equals results/production.json.
def gev(value, channel, sign=False):
    """A GeV value with the channel's decimals, as in the text."""
    decimals = 4 if channel == "Z_mumu" else 5
    if value is None:
        return "n/a"
    return f"{value:+.{decimals}f}" if sign else f"{value:.{decimals}f}"


def stated_numbers(channel):
    """The text's numbers for one channel, recomputed here from the production file."""
    c = PRODUCTION["channels"][channel]
    r = c["result"]
    numbers = {
        "peak position": gev(r["peak_position_gev"], channel),
        "raw stat. error": gev(r["stat_error_gev"], channel),
        "scaled stat. error": gev(r["stat_error_scaled_gev"], channel),
        "scale factor": f"{r['scale_factor']:.3f}",
        "resolution": gev(r["resolution_gev"], channel),
        "resolution error": gev(r["resolution_error_gev"], channel),
        "chi2": f"{r['chi2']:.1f}",
        "ndf": str(r["ndf"]),
        "chi2/ndf": f"{r['chi2_ndf']:.3f}",
        "chi2/ndf benchmark": f"{c['t2']['benchmark']:.1f}",
        "PDG mass": f"{r['pdg_mass_gev']:.4f}",
        "offset": gev(r["offset_from_pdg_gev"], channel, sign=True),
        "fit-choice spread": gev(c["fit_choice_spread_gev"], channel),
        "pairs": f"{r['n_pairs_in_fit_range']:,}",
        "events": f"{r['n_events']:,}",
        "fit range low": f"{r['fit_range_gev'][0]:g}",
        "fit range high": f"{r['fit_range_gev'][1]:g}",
        "files": str(PRODUCTION["sample"]["n_files"]),
        "benchmark files": str(PRODUCTION["targets"]["T4"]["benchmark"]["n_files"]),
    }
    return {f"{channel} {name}": value for name, value in numbers.items()}


notebook = json.loads((ROOT / "notebooks" / NOTEBOOK).read_text())
production_cells = [cell for cell in notebook["cells"] if "production-text" in cell["metadata"].get("tags", [])]
text = "\n".join("".join(cell["source"]) for cell in production_cells)
stated = {}
for channel in NOTEBOOK_CHANNELS:
    stated.update(stated_numbers(channel))
for name, value in stated.items():
    checks.check(f"the text states the {name} of production.json", value.lstrip("+") in text, value)
visible = re.sub(r"\]\([^)]*\)", "]", text)
visible = re.sub(r"<!--.*?-->", "", visible, flags=re.S).replace("Run 2012", "").replace("8 TeV", "")
found = re.findall(r"(?<![\w.])\d[\d,]*(?:\.\d+)?(?![\w])", visible)
known = {value.lstrip("+-") for value in stated.values()}
typed = sorted({n.rstrip(",") for n in found} - known)
checks.check("every number in the text comes from production.json", not typed, ", ".join(typed) or "none other")
for name in NOTEBOOK_FIGURES:
    path = FIGURES / name
    checks.check(f"docs/figures/{name} exists and is listed in production.json",
                 path.exists() and f"docs/figures/{name}" in PRODUCTION["figures"])

In [ ]:
display(Image(filename=str(FIGURES / "jpsi_peak_fit.png"), width=720))
display(Markdown((FIGURES / "jpsi_peak_fit_caption.md").read_text()))
display(Image(filename=str(FIGURES / "jpsi_os_vs_ss.png"), width=720))

The lower panel of the fit shows the pulls. A pattern in them would show where the line shape falls short; read it together with
the χ²/ndf above.

In [ ]:
if checks.failures:
    raise AssertionError(f"{len(checks.failures)} check(s) failed: {', '.join(checks.failures)}")
print("All checks passed")